# LeetCode 635 — Design Log Storage System bằng Python

Link đề: https://leetcode.com/problems/design-log-storage-system/ · Độ khó: **Medium** · Chủ đề: Design, Hash Table, String, Ordered Set

> **Ghi chú Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (câu chữ, ràng buộc, và nhất là **thứ tự** các `id` trả về trong ví dụ).

Skeleton LeetCode:

```python
class LogSystem:

    def __init__(self):
        ...

    def put(self, id: int, timestamp: str) -> None:
        ...

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        ...


# Your LogSystem object will be instantiated and called as such:
# obj = LogSystem()
# obj.put(id,timestamp)
# param_2 = obj.retrieve(start,end,granularity)
```

Ghi chú: bài **thiết kế lớp**. Đề không nói rõ thứ tự các `id` trả về (ví dụ của đề in `[3, 2, 1]` và `[2, 1]`, các lời giải khác nhau trả thứ tự khác nhau mà vẫn được chấp nhận), nên notebook **so sánh tập kết quả sau khi sắp xếp** và tự dựng bộ chạy chuỗi lệnh mô phỏng cách LeetCode gọi lớp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưu các log `(id, timestamp)` và truy vấn các `id` có timestamp nằm trong đoạn `[start, end]` **chỉ xét đến độ mịn (granularity)** cho trước — các trường nhỏ hơn bị bỏ qua.
2. Nắm các công cụ Python: cắt chuỗi `s[:k]`, so sánh chuỗi theo **từ điển**, `str.split(":")`, so sánh `list`/`tuple` theo từ điển, `dict` ánh xạ tên độ mịn → độ dài, `bisect` trên danh sách đã sắp xếp, đổi **cơ số hỗn hợp** (mixed radix) bằng phép nhân/chia, và **trie** bằng dict lồng nhau.
3. Đi từ **tách số + so sánh tuple** → **so sánh tiền tố chuỗi** → **khóa số nguyên + chia lấy phần nguyên** → **danh sách có thứ tự + `bisect`** → **trie duyệt có cờ "sát cận"**.
4. Hiểu mấu chốt: timestamp có **độ dài cố định và đệm số 0**, nên **thứ tự từ điển = thứ tự thời gian**; "bỏ qua các trường nhỏ" chỉ là **cắt tiền tố** cho cả `start`, `end` và log.
5. Tránh các lỗi kinh điển: so cả chuỗi dài bỏ qua granularity, cắt sai độ dài (lệch 1), kiểm tra từng trường độc lập thay vì từ điển, `end` loại trừ thay vì bao gồm, và với `bisect` quên **đệm cận trên** bằng giá trị lớn nhất.

## 2. Đề bài & ví dụ

Có nhiều **log**, mỗi log gồm một `id` duy nhất và một `timestamp` dạng chuỗi `Year:Month:Day:Hour:Minute:Second`, ví dụ `"2017:01:01:23:59:59"`. Mọi trường là số thập phân **đệm số 0** (năm 4 chữ số, các trường còn lại 2 chữ số).

Cài lớp `LogSystem`:

- `LogSystem()`: khởi tạo.
- `put(id, timestamp)`: lưu log `(id, timestamp)`.
- `retrieve(start, end, granularity)`: trả về các `id` của những log có timestamp trong đoạn từ `start` đến `end` (**bao gồm cả hai đầu**). `start`, `end` cùng định dạng với `timestamp`; `granularity` cho biết **độ mịn**: ví dụ `start = "2017:01:01:23:59:59"`, `end = "2017:01:02:23:59:59"`, `granularity = "Day"` nghĩa là lấy các log từ **ngày 1/1/2017** đến **ngày 2/1/2017**, bỏ qua Giờ, Phút, Giây của mỗi log.

**Ví dụ:**

```
ops  = ["LogSystem", "put", "put", "put", "retrieve", "retrieve"]
args = [[], [1, "2017:01:01:23:59:59"], [2, "2017:01:01:22:59:59"], [3, "2016:01:01:00:00:00"],
        ["2016:01:01:01:01:01", "2017:01:01:23:00:00", "Year"],
        ["2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour"]]
Output: [null, null, null, null, [3, 2, 1], [2, 1]]
```

- `Year`: trả về mọi log từ năm 2016 đến năm 2017 → `[3, 2, 1]`.
- `Hour`: từ `2016:01:01:01` đến `2017:01:01:23` → log 3 (`2016:01:01:00`) nằm **trước** đầu đoạn nên bị loại → `[2, 1]`.

**Ràng buộc:** `1 <= id <= 500`; `2000 <= Year <= 2017`; `1 <= Month <= 12`; `1 <= Day <= 31`; `0 <= Hour <= 23`; `0 <= Minute, Second <= 59`; `granularity` thuộc `["Year", "Month", "Day", "Hour", "Minute", "Second"]`; tối đa `500` lần gọi `put` và `retrieve`.

In [1]:
import bisect
import itertools
import random
import time
from typing import List


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp: ops[0] là tên lớp (hàm tạo)."""
    obj, out = None, []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


def norm(out):
    """Chuẩn hóa kết quả để so sánh: danh sách id được sắp xếp (đề không ép thứ tự)."""
    return [sorted(x) if isinstance(x, list) else x for x in out]


ops = ["LogSystem", "put", "put", "put", "retrieve", "retrieve"]
args = [[], [1, "2017:01:01:23:59:59"], [2, "2017:01:01:22:59:59"], [3, "2016:01:01:00:00:00"],
        ["2016:01:01:01:01:01", "2017:01:01:23:00:00", "Year"],
        ["2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour"]]
EXPECTED = [None, None, None, None, [1, 2, 3], [1, 2]]      # đã sắp xếp
for o, a in zip(ops, args):
    print(f"{o:<10} {a}")
print("kết quả mong đợi (sau khi sắp xếp):", EXPECTED)

LogSystem  []
put        [1, '2017:01:01:23:59:59']
put        [2, '2017:01:01:22:59:59']
put        [3, '2016:01:01:00:00:00']
retrieve   ['2016:01:01:01:01:01', '2017:01:01:23:00:00', 'Year']
retrieve   ['2016:01:01:01:01:01', '2017:01:01:23:00:00', 'Hour']
kết quả mong đợi (sau khi sắp xếp): [None, None, None, None, [1, 2, 3], [1, 2]]


## 3. Kiến thức nền

### 3.1. Cắt chuỗi và so sánh theo từ điển

`s[:k]` lấy `k` ký tự đầu. Hai chuỗi so sánh bằng `<`, `<=` theo **thứ tự từ điển**: so ký tự đầu, bằng nhau thì so ký tự tiếp theo... Với chuỗi **cùng độ dài và đệm số 0**, thứ tự từ điển **trùng** thứ tự số: `"05" < "12"` đúng, còn không đệm thì `"5" < "12"` sai (`"5" > "1"`).

In [2]:
ts = "2017:01:01:23:59:59"
print(ts[:4], ts[:7], ts[:10], ts[:13])
print("2016:12:31" < "2017:01:01", "05" < "12", "5" < "12", 5 < 12)
print(ts[:7] == "2017:01", len(ts))

2017 2017:01 2017:01:01 2017:01:01:23
True True False True
True 19


### 3.2. `split` và so sánh `list`/`tuple`

`ts.split(":")` tách timestamp thành 6 chuỗi; `int(x)` đổi thành số. `list` và `tuple` cũng so sánh **theo từ điển**: so phần tử đầu, bằng nhau thì so tiếp. Vì vậy `[2016, 12] < [2017, 1]` là `True`.

In [3]:
parts = [int(x) for x in ts.split(":")]
print(parts)
print([2016, 12] < [2017, 1], [2017, 1, 5] < [2017, 1, 7], [2017, 1] <= [2017, 1, 0])
print(parts[:3], parts[:3] <= [2017, 1, 1])

[2017, 1, 1, 23, 59, 59]
True True True
[2017, 1, 1] True


### 3.3. `dict` ánh xạ tên → số và `list.index`

Ta cần đổi `granularity` (chữ) thành "giữ bao nhiêu phần": `dict` hoặc `list.index`. `FIELDS.index("Day") + 1 = 3` phần (Year, Month, Day).

In [4]:
FIELDS = ["Year", "Month", "Day", "Hour", "Minute", "Second"]
GRAN = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}   # độ dài tiền tố chuỗi
print({g: FIELDS.index(g) + 1 for g in FIELDS})
print(GRAN["Day"], ts[:GRAN["Day"]])
# độ dài tiền tố = 4 + 3*(số phần - 1)
assert all(GRAN[g] == 4 + 3 * FIELDS.index(g) for g in FIELDS)

{'Year': 1, 'Month': 2, 'Day': 3, 'Hour': 4, 'Minute': 5, 'Second': 6}
10 2017:01:01


### 3.4. `bisect` trên danh sách đã sắp xếp

`bisect.bisect_left(a, x)` trả về vị trí chèn `x` (trước các phần tử bằng `x`); `bisect_right` (sau các phần tử bằng `x`), mỗi lần `O(log n)`. `bisect.insort(a, x)` chèn giữ nguyên thứ tự (`O(n)` do dịch phần tử). Các phần tử trong `a[l:r]` giữa hai vị trí chính là một **đoạn liên tiếp**.

In [5]:
a = [10, 20, 20, 30, 40]
print(bisect.bisect_left(a, 20), bisect.bisect_right(a, 20), bisect.bisect_left(a, 25))
l, r = bisect.bisect_left(a, 20), bisect.bisect_right(a, 35)
print(a[l:r])                                           # các phần tử trong [20, 35]
bisect.insort(a, 25)
print(a)
print(bisect.bisect_left([(1, 9), (3, 5)], (3,)))      # tuple ngắn hơn nhỏ hơn tuple dài cùng tiền tố

1 3 3
[20, 20, 30]
[10, 20, 20, 25, 30, 40]
1


### 3.5. Cơ số hỗn hợp (mixed radix): gộp nhiều trường thành một số nguyên

Giống cách `giờ:phút:giây` gộp thành giây: `t = (h * 60 + m) * 60 + s`. Mỗi trường nhân với **kích thước** của trường nhỏ hơn. Nếu mỗi trường **nhỏ hơn kích thước** của nó, số nguyên kết quả **giữ nguyên thứ tự** với bộ trường. Chia nguyên (`//`) cho tích các kích thước còn lại **cắt bỏ trường nhỏ**.

In [6]:
h, m, s = 3, 25, 40
t = (h * 60 + m) * 60 + s
print(t, "giây;  bỏ giây:", t // 60, "= phút tổng", h * 60 + m, ";  bỏ phút+giây:", t // 3600)
# thứ tự được giữ: (1, 59, 59) < (2, 0, 0) cũng là 7199 < 7200
print((1 * 60 + 59) * 60 + 59, (2 * 60 + 0) * 60 + 0)

12340 giây;  bỏ giây: 205 = phút tổng 205 ;  bỏ phút+giây: 3
7199 7200


### 3.6. Trie bằng `dict` lồng nhau

Mỗi timestamp là một "đường đi" gồm 6 mắt xích (`"2017" → "01" → "01" → "23" → "59" → "59"`). `dict` lồng nhau lưu đường đi chung một lần: `node = node.setdefault(key, {})` xuống tầng dưới (tạo nếu chưa có).

In [7]:
root = {}
for stamp in ("2017:01:01", "2017:01:02", "2016:12:31"):
    node = root
    for p in stamp.split(":"):
        node = node.setdefault(p, {})
print(root)

{'2017': {'01': {'01': {}, '02': {}}}, '2016': {'12': {'31': {}}}}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xem **Ví dụ**, truy vấn `Hour` với `start = 2016:01:01:01:01:01`, `end = 2017:01:01:23:00:00`.

**Bỏ qua trường nhỏ** nghĩa là chỉ giữ đến **Giờ** của cả ba bên:

| | giữ đến Giờ |
|---|---|
| `start` | `2016:01:01:01` |
| `end` | `2017:01:01:23` |
| log 1 `2017:01:01:23:59:59` | `2017:01:01:23` |
| log 2 `2017:01:01:22:59:59` | `2017:01:01:22` |
| log 3 `2016:01:01:00:00:00` | `2016:01:01:00` |

Rồi kiểm tra `start' <= log' <= end'`:

- log 1: `2016:01:01:01 <= 2017:01:01:23 <= 2017:01:01:23` ✓ (bằng `end'` vẫn được: bao gồm hai đầu).
- log 2: `2017:01:01:22` ✓.
- log 3: `2016:01:01:00 < 2016:01:01:01` ✗.

Kết quả `{1, 2}` đúng như đề. Với `Year` thì chỉ còn `2016`, `2017`: cả ba log đều trong `[2016, 2017]` → `{1, 2, 3}`.

**Quan sát:** bước "so sánh" luôn là **so sánh các bộ trường từ trái sang phải** (từ điển). Với chuỗi đệm số 0 thì chính là so sánh chuỗi.

## 5. Cách 1 — Brute Force / baseline: tách số, cắt trường, so sánh `list`

**Ý tưởng:** lưu mỗi log dưới dạng `(id, [năm, tháng, ngày, giờ, phút, giây])`. Khi truy vấn: đổi `start`, `end` thành list số, cắt `k = số phần theo granularity`, rồi duyệt **mọi log** và giữ `id` nếu `lo <= parts[:k] <= hi` (so sánh list theo từ điển).

Đây là bản dịch gần nguyên văn của đề ("bỏ các trường nhỏ rồi so sánh"), nên cũng là **oracle độc lập** (ta sẽ có thêm một oracle viết theo cách khác ở mục 12).

In [8]:
class LogSystemParse:
    def __init__(self):
        self.logs = []

    def put(self, id, timestamp):
        self.logs.append((id, [int(x) for x in timestamp.split(":")]))

    def retrieve(self, start, end, granularity):
        k = FIELDS.index(granularity) + 1
        lo = [int(x) for x in start.split(":")][:k]
        hi = [int(x) for x in end.split(":")][:k]
        return [id for id, parts in self.logs if lo <= parts[:k] <= hi]


out = run_ops(LogSystemParse, ops, args)
print(out)
assert norm(out) == EXPECTED

[None, None, None, None, [1, 2, 3], [1, 2]]


**Độ phức tạp:** `put` tốn `O(1)` (cộng việc tách 6 trường). `retrieve` duyệt `n` log, mỗi lần tách (đã tách sẵn) + so sánh `O(k)` với `k ≤ 6` ⇒ **`O(n)`** (hằng số 6). Space `O(n)`.

**Nút thắt:** mỗi `retrieve` phải **đi qua cả `n` log** dù chỉ vài log nằm trong đoạn; và ta biến chuỗi thành số/list một cách rườm rà. Với `n ≤ 500` không sao, nhưng ta muốn (a) bỏ bước tách số (chuỗi đã so sánh được!) và (b) tránh quét hết — nếu log được giữ **đã sắp xếp**, ta chỉ cần tìm hai đầu đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 — chuỗi so sánh được trực tiếp.** Timestamp có định dạng **cố định**: `YYYY:MM:DD:hh:mm:ss`, mọi trường đệm số 0. Với hai timestamp bất kỳ, so sánh **chuỗi** theo từ điển cho kết quả giống so sánh bộ số (vì mỗi trường cùng độ rộng và dấu `:` ở cùng vị trí). Do đó không cần `split`/`int`.

**Bổ đề 2 — "bỏ trường nhỏ" là cắt tiền tố.** Giữ đến trường thứ `j` tương ứng giữ `4 + 3·(j − 1)` ký tự đầu: `Year` 4, `Month` 7, `Day` 10, `Hour` 13, `Minute` 16, `Second` 19. Log thỏa khi:

```
start[:p] <= ts[:p] <= end[:p]        với p = GRAN[granularity]
```

**Bổ đề 3 — đoạn liên tiếp trong danh sách đã sắp xếp.** Nếu giữ các log sắp theo `timestamp`, tập log thỏa điều kiện trên là một **đoạn liên tiếp** (vì điều kiện chỉ phụ thuộc vào tiền tố, mà tiền tố là hàm đơn điệu của chuỗi). Ta tìm hai biên bằng `bisect`: biên dưới `start[:p] + (phần thấp nhất)`, biên trên `end[:p] + (phần cao nhất)` — "phần thấp/cao nhất" chính là **đệm bằng các trường nhỏ nhất/lớn nhất** (`:01:01:00:00:00` / `:12:31:23:59:59`).

Từ đây có các cách: **so tiền tố chuỗi** (Cách 2), **khóa số nguyên + chia** (Cách 3), **danh sách sắp xếp + bisect** (Cách 4), và **trie + cờ sát cận** (Cách 5).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: Cách 1 (mục 5) và bốn cách dưới đây. Về ý tưởng chúng khác nhau thật: Cách 2 so **chuỗi tiền tố**; Cách 3 quy mọi thứ về **một số nguyên** rồi chia; Cách 4 giữ dữ liệu **đã sắp xếp** và tìm biên; Cách 5 tổ chức dữ liệu thành **cây** theo từng trường và chỉ đi vào các nhánh có thể thỏa.

### Cách 2 — So sánh tiền tố chuỗi (cách chính thức)

**Giải thích từng đoạn:**

1. `put`: chỉ `append((id, timestamp))`.
2. `retrieve`: `p = GRAN[granularity]`.
3. Giữ `id` khi `start[:p] <= ts[:p] <= end[:p]`.

In [9]:
class LogSystemSlice:
    def __init__(self):
        self.logs = []
        self.d = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}

    def put(self, id, timestamp):
        self.logs.append((id, timestamp))

    def retrieve(self, start, end, granularity):
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]


out = run_ops(LogSystemSlice, ops, args)
print(out)
assert norm(out) == EXPECTED

[None, None, None, None, [1, 2, 3], [1, 2]]


**Code nộp LeetCode (Cách 2):**

```python
class LogSystem:
    def __init__(self):
        self.logs = []
        self.d = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}

    def put(self, id: int, timestamp: str) -> None:
        self.logs.append((id, timestamp))

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]
```

**Độ phức tạp:** `put` `O(1)`; `retrieve` `O(n·p)` với `p ≤ 19` (so chuỗi) ⇒ `O(n)`. Space `O(n)`.

### Cách 3 — Khóa số nguyên (mixed radix) + chia lấy phần nguyên

**Giải thích từng đoạn:**

1. Đổi timestamp thành **một số nguyên** `key` theo cơ số hỗn hợp với kích thước các trường `SIZES = (10000, 13, 32, 24, 60, 60)`: mỗi trường nhỏ hơn kích thước (`Month ≤ 12 < 13`, `Day ≤ 31 < 32`, `Hour ≤ 23 < 24`, `Minute, Second ≤ 59 < 60`). Nhờ đó `key` **giữ nguyên thứ tự thời gian**.
2. "Bỏ các trường nhỏ" = chia nguyên cho `div[g]` = tích các kích thước **sau** trường `g` (ví dụ `Hour`: chia cho `60·60`).
3. Truy vấn: `key(start)//div <= key(ts)//div <= key(end)//div`.

Ưu điểm: so sánh bằng **một phép so số nguyên** thay vì so chuỗi; hợp khi cần lưu gọn/sắp xếp.

In [10]:
SIZES = (10000, 13, 32, 24, 60, 60)
DIV = {g: 1 for g in FIELDS}
for j, g in enumerate(FIELDS):
    for s in SIZES[j + 1:]:
        DIV[g] *= s
print(DIV)


def ts_key(timestamp):
    k = 0
    for p, size in zip(timestamp.split(":"), SIZES):
        k = k * size + int(p)
    return k


class LogSystemKey:
    def __init__(self):
        self.logs = []

    def put(self, id, timestamp):
        self.logs.append((id, ts_key(timestamp)))

    def retrieve(self, start, end, granularity):
        div = DIV[granularity]
        lo, hi = ts_key(start) // div, ts_key(end) // div
        return [id for id, k in self.logs if lo <= k // div <= hi]


# kiểm tra khóa giữ thứ tự thời gian và bỏ đúng trường
assert ts_key("2017:01:01:23:59:59") > ts_key("2017:01:01:22:59:59") > ts_key("2016:12:31:23:59:59")
assert ts_key("2017:01:01:23:59:59") // DIV["Day"] == ts_key("2017:01:01:00:00:00") // DIV["Day"]
out = run_ops(LogSystemKey, ops, args)
print(out)
assert norm(out) == EXPECTED

{'Year': 35942400, 'Month': 2764800, 'Day': 86400, 'Hour': 3600, 'Minute': 60, 'Second': 1}
[None, None, None, None, [1, 2, 3], [1, 2]]


**Code nộp LeetCode (Cách 3):**

```python
class LogSystem:
    SIZES = (10000, 13, 32, 24, 60, 60)
    FIELDS = ["Year", "Month", "Day", "Hour", "Minute", "Second"]

    def __init__(self):
        self.logs = []
        self.div = {}
        for j, g in enumerate(self.FIELDS):
            d = 1
            for s in self.SIZES[j + 1:]:
                d *= s
            self.div[g] = d

    def _key(self, ts: str) -> int:
        k = 0
        for p, size in zip(ts.split(":"), self.SIZES):
            k = k * size + int(p)
        return k

    def put(self, id: int, timestamp: str) -> None:
        self.logs.append((id, self._key(timestamp)))

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        d = self.div[granularity]
        lo, hi = self._key(start) // d, self._key(end) // d
        return [id for id, k in self.logs if lo <= k // d <= hi]
```

**Độ phức tạp:** `put` `O(1)` (6 phép nhân); `retrieve` `O(n)`. Space `O(n)`.

### Cách 4 — Danh sách sắp xếp + `bisect` (tìm biên thay vì quét)

**Giải thích từng đoạn:**

1. `put`: `bisect.insort(self.logs, (timestamp, id))` — giữ danh sách **luôn sắp xếp** theo `(timestamp, id)`.
2. `retrieve`: `p = GRAN[granularity]`. Tạo hai chuỗi biên **đệm đủ 19 ký tự**:
   - `lo = start[:p] + MIN_TS[p:]` với `MIN_TS = "0000:01:01:00:00:00"` (trường nhỏ nhất);
   - `hi = end[:p] + MAX_TS[p:]` với `MAX_TS = "9999:12:31:23:59:59"` (trường lớn nhất).
3. `l = bisect_left(logs, (lo, -1))`, `r = bisect_right(logs, (hi, +inf))`; kết quả là `logs[l:r]`. Biên `(lo, -1)` nhỏ hơn mọi `(lo, id)` vì `id ≥ 1`; `(hi, inf)` lớn hơn mọi `(hi, id)`.

Vì sao phải đệm `hi` bằng giá trị **lớn nhất**: `end[:p]` ngắn hơn mọi timestamp đầy đủ có cùng tiền tố nên **nhỏ hơn** chúng — nếu dùng thẳng `end[:p]` làm cận trên ta sẽ **bỏ sót** log có cùng tiền tố với `end` (lỗi ở mục 13).

In [11]:
MIN_TS = "0000:01:01:00:00:00"
MAX_TS = "9999:12:31:23:59:59"
INF = float("inf")


class LogSystemBisect:
    def __init__(self):
        self.logs = []                                    # luôn sắp xếp theo (timestamp, id)

    def put(self, id, timestamp):
        bisect.insort(self.logs, (timestamp, id))

    def retrieve(self, start, end, granularity):
        p = GRAN[granularity]
        lo = start[:p] + MIN_TS[p:]
        hi = end[:p] + MAX_TS[p:]
        l = bisect.bisect_left(self.logs, (lo, -1))
        r = bisect.bisect_right(self.logs, (hi, INF))
        return [id for _, id in self.logs[l:r]]


out = run_ops(LogSystemBisect, ops, args)
print(out)                                                # khớp cả thứ tự [3, 2, 1] / [2, 1]? xem dưới
assert norm(out) == EXPECTED
print("lo/hi cho Hour:", "2016:01:01:01" + MIN_TS[13:], "|", "2017:01:01:23" + MAX_TS[13:])

[None, None, None, None, [3, 2, 1], [2, 1]]
lo/hi cho Hour: 2016:01:01:01:00:00 | 2017:01:01:23:59:59


Nhận xét: cách này trả các `id` **theo thứ tự thời gian tăng dần**, nên ở ví dụ cho ra `[3, 2, 1]` và `[2, 1]` — **đúng thứ tự in trong đề**. (Các cách quét theo thứ tự `put` cho `[1, 2, 3]`; vì đề chấp nhận mọi thứ tự, ta so sánh sau khi sắp xếp.)

**Code nộp LeetCode (Cách 4):**

```python
import bisect

class LogSystem:
    GRAN = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}
    MIN_TS = "0000:01:01:00:00:00"
    MAX_TS = "9999:12:31:23:59:59"

    def __init__(self):
        self.logs = []

    def put(self, id: int, timestamp: str) -> None:
        bisect.insort(self.logs, (timestamp, id))

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        p = self.GRAN[granularity]
        lo = start[:p] + self.MIN_TS[p:]
        hi = end[:p] + self.MAX_TS[p:]
        l = bisect.bisect_left(self.logs, (lo, -1))
        r = bisect.bisect_right(self.logs, (hi, float("inf")))
        return [id for _, id in self.logs[l:r]]
```

**Độ phức tạp:** `put` `O(n)` (insort dịch phần tử; với `n ≤ 500` rất nhanh); `retrieve` `O(log n + k)` với `k` là số log trả về. Space `O(n)`. Hợp khi **truy vấn nhiều, ghi ít**.

### Cách 5 — Trie theo từng trường + duyệt có cờ "sát cận" (tight)

**Giải thích từng đoạn:**

1. `put`: đi xuống cây theo 6 trường (`"2017" → "01" → ...`), ở lá lưu danh sách `id` trong khóa đặc biệt `"#"`.
2. `retrieve`: lấy `d` = số trường theo granularity, `lo = start.split(":")[:d]`, `hi = end.split(":")[:d]`. Duyệt DFS từng tầng với hai cờ:
   - `tight_lo`: tiền tố đã chọn **đang bằng** tiền tố của `lo` ⇒ trường ở tầng này **phải ≥ `lo[level]`**; khi chọn trường **lớn hơn** `lo[level]` thì cờ tắt (phần dưới tự do).
   - `tight_hi`: tương tự, trường **phải ≤ `hi[level]`**.
3. Tới tầng `d`: **thu hết** `id` trong cây con (mọi trường nhỏ hơn đều được bỏ qua).

Đây là cùng kiểu "duyệt chữ số có cờ chạm biên" của các bài đếm số (digit DP) — mỗi tầng chỉ có tối đa **hai nhánh sát cận**, còn lại nằm hoàn toàn trong đoạn.

In [12]:
class LogSystemTrie:
    def __init__(self):
        self.root = {}

    def put(self, id, timestamp):
        node = self.root
        for p in timestamp.split(":"):
            node = node.setdefault(p, {})
        node.setdefault("#", []).append(id)

    def retrieve(self, start, end, granularity):
        d = FIELDS.index(granularity) + 1
        lo = start.split(":")[:d]
        hi = end.split(":")[:d]
        out = []

        def collect(node):                                # thu mọi id trong cây con
            for key, child in node.items():
                if key == "#":
                    out.extend(child)
                else:
                    collect(child)

        def dfs(node, level, tight_lo, tight_hi):
            if level == d:
                collect(node)
                return
            for key, child in node.items():
                if key == "#":
                    continue
                if tight_lo and key < lo[level]:
                    continue
                if tight_hi and key > hi[level]:
                    continue
                dfs(child, level + 1, tight_lo and key == lo[level], tight_hi and key == hi[level])

        dfs(self.root, 0, True, True)
        return out


out = run_ops(LogSystemTrie, ops, args)
print(out)
assert norm(out) == EXPECTED

[None, None, None, None, [1, 2, 3], [1, 2]]


**Code nộp LeetCode (Cách 5):**

```python
class LogSystem:
    FIELDS = ["Year", "Month", "Day", "Hour", "Minute", "Second"]

    def __init__(self):
        self.root = {}

    def put(self, id: int, timestamp: str) -> None:
        node = self.root
        for p in timestamp.split(":"):
            node = node.setdefault(p, {})
        node.setdefault("#", []).append(id)

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        d = self.FIELDS.index(granularity) + 1
        lo, hi = start.split(":")[:d], end.split(":")[:d]
        out = []

        def collect(node):
            for key, child in node.items():
                if key == "#":
                    out.extend(child)
                else:
                    collect(child)

        def dfs(node, level, tl, th):
            if level == d:
                collect(node)
                return
            for key, child in node.items():
                if key == "#":
                    continue
                if tl and key < lo[level]:
                    continue
                if th and key > hi[level]:
                    continue
                dfs(child, level + 1, tl and key == lo[level], th and key == hi[level])

        dfs(self.root, 0, True, True)
        return out
```

**Độ phức tạp:** `put` `O(6)`. `retrieve`: ở mỗi tầng ghé qua các con của tối đa hai nút "sát cận" (≤ 18/12/31/24/60/60 con) và thu `k` kết quả ⇒ `O(k + 6·60)` ≈ `O(k)`, không phụ thuộc `n` khi `k` nhỏ. Space `O(n)` (cộng chi phí nút cây).

### Kiểm tra nhanh năm cách trên ví dụ đề

In [13]:
IMPLS = {
    "parse": LogSystemParse,
    "slice": LogSystemSlice,
    "key": LogSystemKey,
    "bisect": LogSystemBisect,
    "trie": LogSystemTrie,
}
for name, cls in IMPLS.items():
    out = run_ops(cls, ops, args)
    assert norm(out) == EXPECTED, name
    print(f"{name:<7}", out[4], out[5])

parse   [1, 2, 3] [1, 2]
slice   [1, 2, 3] [1, 2]
key     [1, 2, 3] [1, 2]
bisect  [3, 2, 1] [2, 1]
trie    [1, 2, 3] [1, 2]


## 8. TRACE từng bước

**Trace Cách 2 trên `Hour`** (`p = 13`): ta in tiền tố của `start`, `end` và từng log, cùng kết luận.

| Log | `ts[:13]` | `start[:13] <= ts[:13]` | `ts[:13] <= end[:13]` | Giữ? |
|---|---|---|---|---|
| 1 `2017:01:01:23:59:59` | `2017:01:01:23` | ✓ | ✓ (bằng) | ✓ |
| 2 `2017:01:01:22:59:59` | `2017:01:01:22` | ✓ | ✓ | ✓ |
| 3 `2016:01:01:00:00:00` | `2016:01:01:00` | ✗ (`00 < 01`) | ✓ | ✗ |

In [14]:
def trace_slice(logs, start, end, granularity):
    p = GRAN[granularity]
    lo, hi = start[:p], end[:p]
    print(f"granularity={granularity} (p={p}): lo={lo!r}, hi={hi!r}")
    result = []
    for id, ts in logs:
        pre = ts[:p]
        a, b = lo <= pre, pre <= hi
        keep = a and b
        print(f"  log {id}: {pre!r:<18} lo<=pre:{a!s:<5} pre<=hi:{b!s:<5} -> {'GIỮ' if keep else 'bỏ'}")
        if keep:
            result.append(id)
    print("  =>", result, "\n")
    return result


logs_ex = [(1, "2017:01:01:23:59:59"), (2, "2017:01:01:22:59:59"), (3, "2016:01:01:00:00:00")]
assert sorted(trace_slice(logs_ex, "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Year")) == [1, 2, 3]
assert sorted(trace_slice(logs_ex, "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour")) == [1, 2]

granularity=Year (p=4): lo='2016', hi='2017'
  log 1: '2017'             lo<=pre:True  pre<=hi:True  -> GIỮ
  log 2: '2017'             lo<=pre:True  pre<=hi:True  -> GIỮ
  log 3: '2016'             lo<=pre:True  pre<=hi:True  -> GIỮ
  => [1, 2, 3] 

granularity=Hour (p=13): lo='2016:01:01:01', hi='2017:01:01:23'
  log 1: '2017:01:01:23'    lo<=pre:True  pre<=hi:True  -> GIỮ
  log 2: '2017:01:01:22'    lo<=pre:True  pre<=hi:True  -> GIỮ
  log 3: '2016:01:01:00'    lo<=pre:False pre<=hi:True  -> bỏ
  => [1, 2] 



**Trace Cách 4 (bisect):** in danh sách đã sắp xếp, hai biên `lo`/`hi` sau khi đệm, và hai vị trí `l`, `r`.

In [15]:
def trace_bisect(logs, start, end, granularity):
    srt = sorted((ts, id) for id, ts in logs)
    p = GRAN[granularity]
    lo, hi = start[:p] + MIN_TS[p:], end[:p] + MAX_TS[p:]
    l = bisect.bisect_left(srt, (lo, -1))
    r = bisect.bisect_right(srt, (hi, INF))
    print("danh sách sắp xếp:")
    for i, (ts, id) in enumerate(srt):
        mark = "<-- trong đoạn" if l <= i < r else ""
        print(f"  [{i}] {ts} id={id} {mark}")
    print(f"lo = {lo}\nhi = {hi}\nl = {l}, r = {r}  -> id = {[id for _, id in srt[l:r]]}\n")
    return [id for _, id in srt[l:r]]


assert trace_bisect(logs_ex, "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour") == [2, 1]
assert trace_bisect(logs_ex, "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Year") == [3, 2, 1]

danh sách sắp xếp:
  [0] 2016:01:01:00:00:00 id=3 
  [1] 2017:01:01:22:59:59 id=2 <-- trong đoạn
  [2] 2017:01:01:23:59:59 id=1 <-- trong đoạn
lo = 2016:01:01:01:00:00
hi = 2017:01:01:23:59:59
l = 1, r = 3  -> id = [2, 1]

danh sách sắp xếp:
  [0] 2016:01:01:00:00:00 id=3 <-- trong đoạn
  [1] 2017:01:01:22:59:59 id=2 <-- trong đoạn
  [2] 2017:01:01:23:59:59 id=1 <-- trong đoạn
lo = 2016:01:01:00:00:00
hi = 2017:12:31:23:59:59
l = 0, r = 3  -> id = [3, 2, 1]



**Trace Cách 5 (trie + cờ sát cận):** in từng bước xuống cây để thấy nhánh nào bị cắt, nhánh nào chỉ còn một cờ.

In [16]:
def trace_trie(logs, start, end, granularity):
    root = {}
    for id, ts in logs:
        node = root
        for p in ts.split(":"):
            node = node.setdefault(p, {})
        node.setdefault("#", []).append(id)
    d = FIELDS.index(granularity) + 1
    lo, hi = start.split(":")[:d], end.split(":")[:d]
    print(f"d={d}, lo={lo}, hi={hi}")
    out = []

    def collect(node):
        for key, child in node.items():
            out.extend(child) if key == "#" else collect(child)

    def dfs(node, level, tl, th):
        ind = "  " * (level + 1)
        if level == d:
            before = len(out)
            collect(node)
            print(f"{ind}thu cây con -> {out[before:]}")
            return
        for key, child in node.items():
            if key == "#":
                continue
            if tl and key < lo[level]:
                print(f"{ind}tầng {level} khóa {key}: CẮT (< {lo[level]} khi sát cận dưới)")
                continue
            if th and key > hi[level]:
                print(f"{ind}tầng {level} khóa {key}: CẮT (> {hi[level]} khi sát cận trên)")
                continue
            ntl, nth = tl and key == lo[level], th and key == hi[level]
            print(f"{ind}tầng {level} khóa {key}: đi xuống (tight_lo={ntl}, tight_hi={nth})")
            dfs(child, level + 1, ntl, nth)

    dfs(root, 0, True, True)
    print("=>", out, "\n")
    return out


assert sorted(trace_trie(logs_ex, "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour")) == [1, 2]

d=4, lo=['2016', '01', '01', '01'], hi=['2017', '01', '01', '23']
  tầng 0 khóa 2017: đi xuống (tight_lo=False, tight_hi=True)
    tầng 1 khóa 01: đi xuống (tight_lo=False, tight_hi=True)
      tầng 2 khóa 01: đi xuống (tight_lo=False, tight_hi=True)
        tầng 3 khóa 23: đi xuống (tight_lo=False, tight_hi=True)
          thu cây con -> [1]
        tầng 3 khóa 22: đi xuống (tight_lo=False, tight_hi=False)
          thu cây con -> [2]
  tầng 0 khóa 2016: đi xuống (tight_lo=True, tight_hi=False)
    tầng 1 khóa 01: đi xuống (tight_lo=True, tight_hi=False)
      tầng 2 khóa 01: đi xuống (tight_lo=True, tight_hi=False)
        tầng 3 khóa 00: CẮT (< 01 khi sát cận dưới)
=> [1, 2] 



## 9. Code nộp LeetCode

Ta nộp **Cách 2** (so sánh tiền tố chuỗi): ngắn, dễ chứng minh, `O(n)` mỗi truy vấn là quá đủ với `n ≤ 500`.

In [17]:
class LogSystem:
    def __init__(self):
        self.logs = []
        self.d = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}

    def put(self, id: int, timestamp: str) -> None:
        self.logs.append((id, timestamp))

    def retrieve(self, start: str, end: str, granularity: str) -> List[int]:
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]


# Your LogSystem object will be instantiated and called as such:
# obj = LogSystem()
# obj.put(id,timestamp)
# param_2 = obj.retrieve(start,end,granularity)
out = run_ops(LogSystem, ops, args)
print(out)
assert norm(out) == EXPECTED
IMPLS["LogSystem"] = LogSystem

[None, None, None, None, [1, 2, 3], [1, 2]]


## 10. Vì sao đúng

- **So sánh chuỗi = so sánh thời gian.** Mọi trường được đệm số 0 với độ rộng cố định (4, 2, 2, 2, 2, 2) và các dấu `:` nằm ở cùng vị trí, nên hai timestamp bất kỳ so sánh từ điển ở **vị trí khác nhau đầu tiên**, đó cũng chính là trường lớn nhất khác nhau — trường quyết định thứ tự thời gian.
- **Tiền tố = "bỏ các trường nhỏ".** Giữ `p = 4 + 3(j−1)` ký tự đầu giữ lại đúng `j` trường đầu. Vì `p` đúng bằng ranh giới trường, hai tiền tố vẫn là chuỗi cùng độ dài (độ rộng cố định), nên bất đẳng thức `start[:p] <= ts[:p] <= end[:p]` chính là "log nằm trong đoạn khi chỉ xét `j` trường đầu", **bao gồm** hai đầu như đề yêu cầu. (Nếu `start[:p] > end[:p]` thì đoạn rỗng và không log nào thỏa.)
- **Cách 3:** cơ số hỗn hợp với mỗi trường nhỏ hơn kích thước của nó là **song ánh tăng** từ bộ trường sang số nguyên; chia nguyên cho tích kích thước các trường sau cắt đúng các trường nhỏ. Thứ tự giữ nguyên.
- **Cách 4:** tập log thỏa là đoạn liên tiếp của danh sách sắp xếp vì điều kiện chỉ phụ thuộc vào tiền tố, mà tiền tố đơn điệu theo chuỗi. Log thỏa `⇔ lo <= ts <= hi` với `lo = start[:p] + trường nhỏ nhất`, `hi = end[:p] + trường lớn nhất`: ví dụ `ts[:p] >= start[:p] ⇔ ts >= start[:p] + MIN_TS[p:]` vì mọi `ts` có đủ trường đều không nhỏ hơn phần đệm nhỏ nhất, và `ts[:p] <= end[:p] ⇔ ts <= end[:p] + MAX_TS[p:]` vì mọi trường đều không lớn hơn mức lớn nhất (ràng buộc `Day ≤ 31`, `Month ≤ 12`, ...).
- **Cách 5:** DFS có cờ liệt kê đúng các tiền tố `x` với `lo ≤ x ≤ hi` (từ điển): ở tầng đầu tiên, trường `> lo` và `< hi` thì mọi hậu tố đều hợp lệ (cờ tắt); trường `= lo` giữ cờ dưới; `= hi` giữ cờ trên. Cả hai cờ có thể cùng bật (khi `lo` và `hi` chung tiền tố).
- **`id`:** mỗi `put` lưu một log và không bao giờ sửa/xóa, nên `retrieve` luôn thấy đủ các log đã `put`.

Mục 12 đối chiếu cả năm cách với một oracle độc lập trên toàn bộ không gian nhỏ và hàng nghìn chuỗi lệnh ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số log đã lưu, `k` là số log trả về.

| Cách | `put` | `retrieve` | Space | Ghi chú |
|---|---|---|---|---|
| 1. Tách số + so sánh list | `O(1)` | `O(n)` | `O(n)` | oracle dễ hiểu |
| 2. So tiền tố chuỗi | `O(1)` | `O(n)` | `O(n)` | **nộp bài** |
| 3. Khóa số nguyên + chia | `O(1)` | `O(n)` | `O(n)` | so sánh bằng số |
| 4. Sắp xếp + `bisect` | `O(n)` (insort) | `O(log n + k)` | `O(n)` | truy vấn nhanh, ghi chậm hơn |
| 5. Trie + cờ sát cận | `O(1)` | `O(k + hằng)` | `O(n)` | không phụ thuộc `n` khi `k` nhỏ |

Ràng buộc `≤ 500` lệnh nên mọi cách đều tức thì; sự khác biệt chỉ lộ khi ta thử vượt xa ràng buộc (mục 12.7). Lưu ý bài đo ở đó dùng các đoạn ngẫu nhiên trải rộng nên `k` rất lớn: `bisect` thắng nhờ cắt lát danh sách, còn trie chậm nhất vì phải đệ quy Python để thu từng cây con — ưu thế của trie chỉ xuất hiện khi `k` nhỏ so với `n`.

## 12. Bộ test

Vì đề **không ép thứ tự** `id`, ta so sánh **tập kết quả (sau khi sắp xếp)**. Chiến lược:

1. **Ví dụ đề** qua bộ chạy chuỗi lệnh `run_ops` cho cả 5 cách + `LogSystem`.
2. **Oracle độc lập** `oracle(...)`: viết theo cách khác hẳn — "**đưa các trường nhỏ về giá trị nhỏ nhất rồi so sánh timestamp đầy đủ**" (không dùng tiền tố chuỗi, không `bisect`).
3. **Edge case:** đúng biên (log bằng `start`/`end`), `start > end` (rỗng), cùng timestamp nhiều `id`, `retrieve` khi chưa có log, mỗi granularity, trường `31`/`12`/`59` ở cận trên.
4. **Vét cạn:** một lưới 64 timestamp (mỗi trường chỉ lấy giá trị nhỏ nhất và lớn nhất) × mọi cặp `(start, end)` × 6 granularity.
5. **Ngẫu nhiên có seed:** chuỗi lệnh đúng ràng buộc (≤ 500 lệnh, `id` duy nhất 1..500), timestamp **tập trung** (ít năm/tháng) để có nhiều khớp.
6. **Thời gian** vượt xa ràng buộc để thấy khác biệt các cách.

In [18]:
MINS = [0, 1, 1, 0, 0, 0]


def oracle(logs, start, end, granularity):
    """logs: danh sách (id, timestamp). Đưa các trường nhỏ hơn độ mịn về giá trị nhỏ nhất rồi so sánh đầy đủ."""
    k = FIELDS.index(granularity) + 1

    def floor_ts(ts):
        parts = [int(x) for x in ts.split(":")]
        return tuple(parts[:k] + MINS[k:])

    lo, hi = floor_ts(start), floor_ts(end)
    return sorted(id for id, ts in logs if lo <= floor_ts(ts) <= hi)


# 1) Ví dụ đề
for name, cls in IMPLS.items():
    assert norm(run_ops(cls, ops, args)) == EXPECTED, name
assert oracle([(1, "2017:01:01:23:59:59"), (2, "2017:01:01:22:59:59"), (3, "2016:01:01:00:00:00")],
              "2016:01:01:01:01:01", "2017:01:01:23:00:00", "Hour") == [1, 2]
print("Ví dụ đề (5 cách + LogSystem + oracle): OK")

Ví dụ đề (5 cách + LogSystem + oracle): OK


In [19]:
# 2-3) Edge case
def session(cls, puts, queries):
    obj = cls()
    for id, ts in puts:
        obj.put(id, ts)
    return [sorted(obj.retrieve(s, e, g)) for s, e, g in queries]


puts = [(1, "2017:01:01:23:59:59"), (2, "2017:01:01:22:59:59"), (3, "2016:01:01:00:00:00"),
        (4, "2016:01:01:00:00:00"),                        # cùng timestamp với log 3
        (5, "2000:12:31:00:00:00"), (6, "2017:12:31:23:59:59")]
queries = [
    ("2016:01:01:00:00:00", "2016:01:01:00:00:00", "Second"),   # đoạn một điểm, bằng cả hai đầu
    ("2016:01:01:00:00:00", "2016:01:01:00:00:00", "Year"),
    ("2017:01:01:23:59:59", "2016:01:01:00:00:00", "Year"),     # start > end: rỗng
    ("2000:01:01:00:00:00", "2017:12:31:23:59:59", "Second"),   # phủ tất cả
    ("2017:01:01:00:00:00", "2017:01:01:00:00:00", "Day"),      # chỉ ngày 1/1/2017
    ("2017:12:01:00:00:00", "2017:12:01:00:00:00", "Month"),    # tháng 12/2017: trường 31/23/59 ở cận trên
    ("2017:01:01:23:59:59", "2017:01:01:23:59:59", "Hour"),
    ("2017:01:01:22:00:00", "2017:01:01:22:00:00", "Hour"),
    ("2018:01:01:00:00:00", "2019:01:01:00:00:00", "Year"),     # hoàn toàn sau mọi log
]
expected_edge = [[3, 4], [3, 4], [], [1, 2, 3, 4, 5, 6], [1, 2], [6], [1], [2], []]
assert session(LogSystemParse, puts, queries) == expected_edge
for name, cls in IMPLS.items():
    assert session(cls, puts, queries) == expected_edge, name
    assert session(cls, [], queries) == [[] for _ in queries], name     # chưa có log nào
orc = [oracle(puts, s, e, g) for s, e, g in queries]
assert orc == expected_edge
print("Edge case: OK ( 9 truy vấn x 5 cách + oracle )")

Edge case: OK ( 9 truy vấn x 5 cách + oracle )


In [20]:
# 4) Vét cạn: lưới 2^6 = 64 timestamp, mọi cặp (start, end), 6 granularity
vals = [("2016", "2017"), ("01", "12"), ("01", "31"), ("00", "23"), ("00", "59"), ("00", "59")]
grid = [":".join(c) for c in itertools.product(*vals)]
assert len(grid) == 64
grid_logs = [(i + 1, ts) for i, ts in enumerate(grid)]

objs = {}
for name, cls in IMPLS.items():
    o = cls()
    for id, ts in grid_logs:
        o.put(id, ts)
    objs[name] = o

checked = 0
for s in grid:
    for e in grid:
        for g in FIELDS:
            want = oracle(grid_logs, s, e, g)
            for name, o in objs.items():
                assert sorted(o.retrieve(s, e, g)) == want, (name, s, e, g)
            checked += 1
print(f"Vét cạn: {checked} truy vấn x 5 cách trùng oracle")

Vét cạn: 24576 truy vấn x 5 cách trùng oracle


In [21]:
# 5) Ngẫu nhiên có seed: chuỗi lệnh đúng ràng buộc
def gen_ts(rng, years, months):
    return "{:04d}:{:02d}:{:02d}:{:02d}:{:02d}:{:02d}".format(
        rng.choice(years), rng.choice(months), rng.randint(1, 31),
        rng.randint(0, 23), rng.randint(0, 59), rng.randint(0, 59))


def gen_session(rng):
    calls = rng.randint(1, 500)
    ids = rng.sample(range(1, 501), calls)
    if rng.random() < 0.5:
        years, months = list(range(2000, 2018)), list(range(1, 13))
    else:                                                  # tập trung: nhiều log cùng năm/tháng
        years, months = rng.sample(range(2000, 2018), 2), rng.sample(range(1, 13), 2)
    ops_, args_ = ["LogSystem"], [[]]
    put_logs = []
    used = 0
    for _ in range(calls):
        if rng.random() < 0.6 and used < len(ids):
            ts = gen_ts(rng, years, months)
            ops_.append("put"); args_.append([ids[used], ts]); put_logs.append((ids[used], ts))
            used += 1
        else:
            s, e = gen_ts(rng, years, months), gen_ts(rng, years, months)
            if rng.random() < 0.8 and s > e:
                s, e = e, s                               # phần lớn truy vấn có start <= end
            if put_logs and rng.random() < 0.3:           # dùng đúng timestamp có sẵn làm biên
                s = rng.choice(put_logs)[1]
            ops_.append("retrieve"); args_.append([s, e, rng.choice(FIELDS)])
    return ops_, args_


def oracle_run(ops_, args_):
    out, logs = [None], []
    for op, a in zip(ops_[1:], args_[1:]):
        if op == "put":
            logs.append((a[0], a[1])); out.append(None)
        else:
            out.append(oracle(logs, *a))
    return out


rng = random.Random(635)
sessions = nonempty = queries_total = 0
for _ in range(400):
    ops_, args_ = gen_session(rng)
    assert len(ops_) - 1 <= 500
    want = oracle_run(ops_, args_)
    for name, cls in IMPLS.items():
        assert norm(run_ops(cls, ops_, args_)) == want, name
    sessions += 1
    for w, op in zip(want, ops_):
        if op == "retrieve":
            queries_total += 1
            nonempty += bool(w)
print(f"Ngẫu nhiên: {sessions} phiên, {queries_total} truy vấn ({nonempty} truy vấn có kết quả) x 5 cách: OK")
assert nonempty > 500

Ngẫu nhiên: 400 phiên, 41119 truy vấn (33096 truy vấn có kết quả) x 5 cách: OK


In [22]:
# 6) Tính chất phụ: kết quả của granularity thô là bao hàm kết quả của granularity mịn (cùng start, end)
rng = random.Random(7)
for _ in range(200):
    logs = [(i + 1, gen_ts(rng, [2016, 2017], [1, 2, 12])) for i in range(40)]
    s, e = sorted([gen_ts(rng, [2016, 2017], [1, 2, 12]) for _ in range(2)])
    obj = LogSystem()
    for id, ts in logs:
        obj.put(id, ts)
    sets = [set(obj.retrieve(s, e, g)) for g in FIELDS]      # Year ... Second
    for coarse, fine in zip(sets, sets[1:]):
        assert fine <= coarse, "granularity mịn hơn thì kết quả không được nhiều hơn"
print("Tính chất: kết quả Year ⊇ Month ⊇ Day ⊇ Hour ⊇ Minute ⊇ Second (với cùng start <= end): OK")

# khi start = end, kết quả = mọi log có cùng tiền tố
for _ in range(100):
    logs = [(i + 1, gen_ts(rng, [2016, 2017], [1, 2])) for i in range(30)]
    s = logs[0][1]
    obj = LogSystem()
    for id, ts in logs:
        obj.put(id, ts)
    for g in FIELDS:
        p = GRAN[g]
        assert sorted(obj.retrieve(s, s, g)) == sorted(id for id, ts in logs if ts[:p] == s[:p])
print("Tính chất: start = end cho đúng các log cùng tiền tố: OK")

Tính chất: kết quả Year ⊇ Month ⊇ Day ⊇ Hour ⊇ Minute ⊇ Second (với cùng start <= end): OK
Tính chất: start = end cho đúng các log cùng tiền tố: OK


In [23]:
# 7) Thời gian (VƯỢT ràng buộc đề để thấy khác biệt): N log, Q truy vấn
def bench(cls, N, Q, seed=1):
    rng = random.Random(seed)
    o = cls()
    t0 = time.perf_counter()
    for i in range(N):
        o.put(i + 1, gen_ts(rng, list(range(2000, 2018)), list(range(1, 13))))
    t_put = (time.perf_counter() - t0) * 1000
    qs = []
    for _ in range(Q):
        a, b = sorted([gen_ts(rng, list(range(2000, 2018)), list(range(1, 13))) for _ in range(2)])
        qs.append((a, b, rng.choice(["Day", "Hour", "Minute"])))
    t0 = time.perf_counter()
    total = sum(len(o.retrieve(*q)) for q in qs)
    t_q = (time.perf_counter() - t0) * 1000
    return t_put, t_q, total


for N in (500, 5000, 20000):
    row = []
    sig = set()
    for name, cls in IMPLS.items():
        if name == "LogSystem":
            continue
        tp, tq, total = bench(cls, N, 200)
        sig.add(total)
        row.append(f"{name} put {tp:6.1f} / q {tq:7.1f} ms")
    assert len(sig) == 1, "mọi cách phải cho cùng tổng số kết quả"
    print(f"N={N:>6}: " + " | ".join(row))

N=   500: parse put    4.0 / q    19.4 ms | slice put    3.2 / q    23.6 ms | key put    2.2 / q    11.9 ms | bisect put    3.5 / q     1.8 ms | trie put    2.6 / q    44.9 ms


N=  5000: parse put   20.6 / q   105.9 ms | slice put   15.4 / q   139.8 ms | key put   21.8 / q    77.0 ms | bisect put   20.8 / q     6.6 ms | trie put   44.4 / q   349.8 ms


N= 20000: parse put   81.8 / q   558.2 ms | slice put   63.4 / q   637.1 ms | key put   86.3 / q   289.5 ms | bisect put   99.9 / q    30.3 ms | trie put   98.0 / q  2254.2 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: so sánh cả chuỗi timestamp, **bỏ qua granularity**

```python
return [id for id, ts in self.logs if start <= ts <= end]
```

Vì sao sai: `granularity` bị phớt lờ nên `Year` cũng bị đối xử như `Second`. Ở ví dụ `Year`, log `3` (`2016:01:01:00:00:00`) nhỏ hơn `start` (`2016:01:01:01:01:01`) và log `1` (`2017:01:01:23:59:59`) lớn hơn `end` (`2017:01:01:23:00:00`) nên cả hai bị loại, trong khi đề đòi lấy cả hai năm 2016 và 2017.

In [24]:
class WrongNoGranularity(LogSystemSlice):
    def retrieve(self, start, end, granularity):
        return [id for id, ts in self.logs if start <= ts <= end]


out = run_ops(WrongNoGranularity, ops, args)
print("sai :", norm(out)[4], "(đáng lẽ [1, 2, 3])")
assert norm(out)[4] == [2] != EXPECTED[4]
print("đúng:", norm(run_ops(LogSystem, ops, args))[4])

sai : [2] (đáng lẽ [1, 2, 3])
đúng: [1, 2, 3]


### Lỗi 2: cắt tiền tố **sai độ dài** (lệch 1 ký tự)

`Month` cần `7` ký tự (`"2017:01"`). Cắt `6` ký tự (`"2017:0"`) chỉ so **chữ số đầu của tháng**, nên tháng `01` và `09` bị coi là bằng nhau.

In [25]:
class WrongPrefixLen(LogSystemSlice):
    def __init__(self):
        super().__init__()
        self.d = {"Year": 4, "Month": 6, "Day": 9, "Hour": 12, "Minute": 15, "Second": 19}   # lệch 1 (trừ Second)


bad, good = WrongPrefixLen(), LogSystem()
for o in (bad, good):
    o.put(1, "2017:01:15:00:00:00")
    o.put(2, "2017:09:15:00:00:00")
q = ("2017:01:01:00:00:00", "2017:02:01:00:00:00", "Month")      # chỉ tháng 1 và 2/2017
print("sai :", sorted(bad.retrieve(*q)), "| đúng:", sorted(good.retrieve(*q)))
assert sorted(bad.retrieve(*q)) == [1, 2] and sorted(good.retrieve(*q)) == [1]

sai : [1, 2] | đúng: [1]


### Lỗi 3: kiểm tra **từng trường độc lập** thay vì so sánh từ điển

Điều kiện đúng là `[năm, tháng] ∈ [[năm_s, tháng_s], [năm_e, tháng_e]]` theo thứ tự từ điển, chứ **không** phải "năm trong đoạn năm **và** tháng trong đoạn tháng". Đoạn từ `2016:12` đến `2017:01` có `tháng_s = 12 > tháng_e = 01`, kiểm tra độc lập sẽ cho tập rỗng.

In [26]:
class WrongPerField(LogSystemParse):
    def retrieve(self, start, end, granularity):
        k = FIELDS.index(granularity) + 1
        lo = [int(x) for x in start.split(":")][:k]
        hi = [int(x) for x in end.split(":")][:k]
        return [id for id, parts in self.logs
                if all(lo[j] <= parts[j] <= hi[j] for j in range(k))]


bad, good = WrongPerField(), LogSystem()
for o in (bad, good):
    o.put(1, "2016:12:15:00:00:00")
    o.put(2, "2017:01:10:00:00:00")
q = ("2016:12:01:00:00:00", "2017:01:31:00:00:00", "Month")
print("sai :", sorted(bad.retrieve(*q)), "| đúng:", sorted(good.retrieve(*q)))
assert sorted(bad.retrieve(*q)) == [] and sorted(good.retrieve(*q)) == [1, 2]

sai : [] | đúng: [1, 2]


### Lỗi 4: dùng `<` thay vì `<=` ở đầu `end` (đoạn mở thay vì **bao gồm**)

Đề nói "**inclusive**": log có cùng tiền tố với `end` vẫn phải được lấy.

In [27]:
class WrongExclusiveEnd(LogSystemSlice):
    def retrieve(self, start, end, granularity):
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] < end[:p]]       # sai: < thay vì <=


out = run_ops(WrongExclusiveEnd, ops, args)
print("sai :", norm(out)[4:], "| đúng:", norm(run_ops(LogSystem, ops, args))[4:])
assert norm(out)[4] == [3] and norm(out)[5] == [2]          # log 1 (cùng tiền tố với end) bị mất

sai : [[3], [2]] | đúng: [[1, 2, 3], [1, 2]]


### Lỗi 5 (với `bisect`): dùng thẳng `end[:p]` làm **cận trên** mà không đệm giá trị lớn nhất

`end[:p]` **ngắn hơn** mọi timestamp đầy đủ cùng tiền tố, nên **nhỏ hơn** chúng (chuỗi ngắn là tiền tố thì nhỏ hơn). `bisect_right(logs, end[:p])` vì thế dừng **trước** các log thuộc cùng giờ/ngày với `end` — bỏ sót chúng.

In [28]:
class WrongBisectNoPad(LogSystemBisect):
    def retrieve(self, start, end, granularity):
        p = GRAN[granularity]
        l = bisect.bisect_left(self.logs, (start[:p], -1))
        r = bisect.bisect_right(self.logs, (end[:p], INF))          # sai: thiếu + MAX_TS[p:]
        return [id for _, id in self.logs[l:r]]


out = run_ops(WrongBisectNoPad, ops, args)
print("sai :", out[4:], "| đúng:", run_ops(LogSystemBisect, ops, args)[4:])
assert sorted(out[5]) == [2] and sorted(run_ops(LogSystemBisect, ops, args)[5]) == [1, 2]

sai : [[3], [2]] | đúng: [[3, 2, 1], [2, 1]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** định dạng ngày giờ **cố định và đệm số 0** ⇒ **so sánh chuỗi = so sánh thời gian**; không cần `datetime` (nhất là khi đề cho phép `Day = 31` ở tháng bất kỳ, `datetime` sẽ báo lỗi ngày không hợp lệ).
- **"Bỏ các trường nhỏ" = cắt tiền tố.** Ghi nhớ độ dài tiền tố: `4, 7, 10, 13, 16, 19` (tăng 3 mỗi bậc sau năm).
- **Hai biên đệm khi dùng `bisect`:** biên dưới đệm **nhỏ nhất** (`:01:01:00:00:00`), biên trên đệm **lớn nhất** (`:12:31:23:59:59`).
- **Chọn cấu trúc theo tỉ lệ ghi/đọc:** nhiều truy vấn → giữ dữ liệu đã sắp xếp / cây; nhiều ghi → danh sách thô + quét. Với `≤ 500` lệnh, quét đơn giản là tốt nhất.
- **Bài thiết kế có đáp án không duy nhất (thứ tự trả về):** test bằng **so sánh tập đã sắp xếp** và một **oracle viết theo cách khác**.
- **Cùng họ:** *Time Based Key-Value Store* (981, `bisect` theo timestamp), *Logger Rate Limiter* (359), *Design Hit Counter* (362), *My Calendar I* (729), *Range Module* (715).
- **Digit-DP/cờ sát cận** (Cách 5) là công cụ chung cho mọi bài "đếm/duyệt các số/chuỗi trong đoạn `[L, R]`".

## 15. Biến thể của bài

1. **Xóa log** (`remove(id)`): với `bisect` cần biết `timestamp` của `id` — giữ thêm dict `id → timestamp`.
2. **Đếm** số log trong đoạn thay vì liệt kê: dùng `r − l` của `bisect` (không cần tạo danh sách).
3. **Khoảng mở/nửa mở** (`[start, end)`): đổi `bisect_right` thành `bisect_left` ở biên trên.
4. **Granularity mới** (`Week`, `Quarter`, `Millisecond`): thêm trường/ánh xạ; `Week` cần chuyển sang ngày thứ tự trong năm.
5. **Cùng `id` được `put` nhiều lần** (cập nhật timestamp).
6. **Truy vấn theo granularity nhưng trả về đã sắp xếp theo thời gian rồi theo `id`.**

Cài nhanh biến thể 1 + 2 trên nền Cách 4 và kiểm tra với oracle:

In [29]:
class LogSystemRemovable:
    def __init__(self):
        self.logs = []                                    # (timestamp, id) luôn sắp xếp
        self.ts_of = {}                                   # id -> timestamp

    def put(self, id, timestamp):
        if id in self.ts_of:                              # cập nhật: bỏ bản cũ
            self.remove(id)
        self.ts_of[id] = timestamp
        bisect.insort(self.logs, (timestamp, id))

    def remove(self, id):
        ts = self.ts_of.pop(id)
        i = bisect.bisect_left(self.logs, (ts, id))
        assert self.logs[i] == (ts, id)
        del self.logs[i]

    def _bounds(self, start, end, granularity):
        p = GRAN[granularity]
        lo, hi = start[:p] + MIN_TS[p:], end[:p] + MAX_TS[p:]
        return bisect.bisect_left(self.logs, (lo, -1)), bisect.bisect_right(self.logs, (hi, INF))

    def retrieve(self, start, end, granularity):
        l, r = self._bounds(start, end, granularity)
        return [id for _, id in self.logs[l:r]]

    def count(self, start, end, granularity):
        l, r = self._bounds(start, end, granularity)
        return max(0, r - l)


rng = random.Random(15)
live = {}
obj = LogSystemRemovable()
for step in range(3000):
    c = rng.random()
    if c < 0.5:
        id = rng.randint(1, 120)
        ts = gen_ts(rng, [2016, 2017], [1, 2, 12])
        obj.put(id, ts); live[id] = ts
    elif c < 0.65 and live:
        id = rng.choice(list(live))
        obj.remove(id); del live[id]
    else:
        s, e = sorted([gen_ts(rng, [2016, 2017], [1, 2, 12]) for _ in range(2)])
        g = rng.choice(FIELDS)
        want = oracle(list(live.items()), s, e, g)
        assert sorted(obj.retrieve(s, e, g)) == want
        assert obj.count(s, e, g) == len(want)
print("Biến thể remove/count (3000 thao tác ngẫu nhiên) khớp oracle: OK;  số log còn:", len(live))

Biến thể remove/count (3000 thao tác ngẫu nhiên) khớp oracle: OK;  số log còn: 83


## 16. So sánh các cách làm

| Cách | Ý tưởng | `put` | `retrieve` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Tách số + so list | dịch nguyên văn đề | `O(1)` | `O(n)` | `O(n)` | làm oracle / bản nháp đầu tiên |
| 2. So tiền tố chuỗi | chuỗi đệm 0 so sánh được, cắt tiền tố | `O(1)` | `O(n)` | `O(n)` | **làm bài** (ngắn, ít lỗi) |
| 3. Khóa số + chia | cơ số hỗn hợp, `//` cắt trường | `O(1)` | `O(n)` | `O(n)` | khi cần lưu/sắp xếp bằng số, chạy trên cột số |
| 4. Sắp xếp + `bisect` | đoạn liên tiếp, đệm hai biên | `O(n)` | `O(log n + k)` | `O(n)` | nhiều truy vấn, ít ghi; cần đếm nhanh |
| 5. Trie + cờ sát cận | cây theo trường, chỉ đi nhánh có thể thỏa | `O(1)` | `O(k + hằng)` | `O(n)` nút | dữ liệu rất lớn, ghi nhiều, truy vấn nhiều |

Trực giác chọn: bài nhỏ → **cắt tiền tố** (Cách 2). Muốn truy vấn nhanh → **sắp xếp + `bisect`** (nhớ đệm hai biên). Cần cấu trúc mở rộng được → **trie**.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

```python
ts = "2016:12:31:23:59:59"
print(ts[:7], ts[:10] < "2017:01:01", ts[:4] <= "2016")
print("2016:12" < "2017:01", "2016:9" < "2016:10")
```

In [30]:
ts_b1 = "2016:12:31:23:59:59"
print(ts_b1[:7], ts_b1[:10] < "2017:01:01", ts_b1[:4] <= "2016")
print("2016:12" < "2017:01", "2016:9" < "2016:10")

2016:12 True True
True False


### Bài 2 (tính tay)

Có các log `(1, "2017:03:05:10:00:00")`, `(2, "2017:03:05:23:59:59")`, `(3, "2017:03:06:00:00:00")`, `(4, "2017:04:01:12:30:00")`. Hỏi: `retrieve("2017:03:05:12:00:00", "2017:03:06:08:00:00", g)` trả về **tập id** nào với `g = "Day"` và với `g = "Hour"`? Ghi vào `my_day` và `my_hour` (kiểu `list` đã sắp xếp; để `None` đến khi tính xong).

In [31]:
logs_b2 = [(1, "2017:03:05:10:00:00"), (2, "2017:03:05:23:59:59"),
           (3, "2017:03:06:00:00:00"), (4, "2017:04:01:12:30:00")]
my_day = None        # TODO: tập id (list đã sắp xếp) khi granularity = "Day"
my_hour = None       # TODO: tập id (list đã sắp xếp) khi granularity = "Hour"

### Bài 3 (điền TODO)

Hoàn thành `retrieve` của lớp dưới đây theo Cách 2: bạn cần (a) lấy độ dài tiền tố `p` từ `self.d`, (b) trả về các `id` thỏa điều kiện. Cell chỉ **định nghĩa**, không gọi.

In [32]:
class LogSystemTodo:
    def __init__(self):
        self.logs = []
        self.d = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}

    def put(self, id, timestamp):
        self.logs.append((id, timestamp))

    def retrieve(self, start, end, granularity):
        # TODO: p = ... ; return [... for id, ts in self.logs if ...]
        pass

### Bài 4 (tự viết lại)

Tự viết `class LogSystemKeyMine` theo **Cách 3** (khóa số nguyên cơ số hỗn hợp): `put` lưu `(id, key)`, `retrieve` chia cho `div[granularity]` rồi so sánh. Không nhìn code cũ. Cell dưới chỉ để khung rỗng.

In [33]:
class LogSystemKeyMine:
    # TODO: viết __init__, put, retrieve (dùng SIZES / DIV ở mục 7 hoặc tự tính lại)
    pass

### Bài 5 (biến thể / mini challenge)

Viết lớp `LogCounter` có `put(id, timestamp)` và `count(start, end, granularity)` trả về **số** log trong đoạn, **không tạo danh sách kết quả** — dùng `bisect` trên danh sách sắp xếp (đệm hai biên). Gợi ý: kết quả là `r - l`.

In [34]:
class LogCounter:
    # TODO: viết __init__, put, count
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `ts[:7]` giữ 7 ký tự; chuỗi so sánh ký tự từng cái một từ trái sang.
- Bài 2: với `Day` bỏ giờ phút giây; với `Hour` giữ đến giờ.
- Bài 3: độ dài tiền tố là `self.d[granularity]`.
- Bài 4: mỗi trường nhân với kích thước của trường nhỏ hơn, cộng vào.
- Bài 5: cần danh sách `(timestamp, id)` luôn sắp xếp.

**Mức 2 (vừa):**
- Bài 1: `"2016:9"` có chữ số `9` đứng sau `"2016:"` còn `"2016:10"` có `1` — so `9` với `1`.
- Bài 2: `Day`: `start' = 2017:03:05`, `end' = 2017:03:06`; xét ngày của từng log. `Hour`: `start' = 2017:03:05:12`, `end' = 2017:03:06:08`.
- Bài 3: `p = self.d[granularity]` rồi list comprehension như Cách 2.
- Bài 4: `div[g] = tích kích thước các trường sau g`; so `lo <= k // div <= hi`.
- Bài 5: `lo = start[:p] + MIN_TS[p:]`, `hi = end[:p] + MAX_TS[p:]`; `bisect_left` và `bisect_right`.

**Mức 3 (gần đáp án):**
- Bài 1: dòng 1 in `2016:12 True True`; dòng 2 in `True False`.
- Bài 2: `Day`: log 1, 2 (ngày 5/3), 3 (ngày 6/3) → `[1, 2, 3]`; log 4 là tháng 4 → loại. `Hour`: log 1 (10h) `< 12h` → loại; log 2 (23h ngày 5) ✓; log 3 (00h ngày 6) ✓ (`≤ 08h`); → `[2, 3]`.
- Bài 3: `return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]`.
- Bài 4: `SIZES = (10000, 13, 32, 24, 60, 60)`, `key = ((((y*13+mo)*32+d)*24+h)*60+mi)*60+s`.
- Bài 5: `return max(0, r - l)` (khi `start > end` có thể `r < l`).

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [35]:
# Bài 1
assert (ts_b1[:7], ts_b1[:10] < "2017:01:01", ts_b1[:4] <= "2016") == ("2016:12", True, True)
assert ("2016:12" < "2017:01", "2016:9" < "2016:10") == (True, False)
print("Bài 1: 2016:12 True True / True False")

# Bài 2
ans_day, ans_hour = [1, 2, 3], [2, 3]
q_start, q_end = "2017:03:05:12:00:00", "2017:03:06:08:00:00"
assert oracle(logs_b2, q_start, q_end, "Day") == ans_day
assert oracle(logs_b2, q_start, q_end, "Hour") == ans_hour
for cls in IMPLS.values():
    o = cls()
    for id, ts in logs_b2:
        o.put(id, ts)
    assert sorted(o.retrieve(q_start, q_end, "Day")) == ans_day and sorted(o.retrieve(q_start, q_end, "Hour")) == ans_hour
print("Bài 2: Day ->", ans_day, "| Hour ->", ans_hour)


# Bài 3
class LogSystemTodoSol(LogSystemTodo):
    def retrieve(self, start, end, granularity):
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]


assert norm(run_ops(LogSystemTodoSol, ops, args)) == EXPECTED
print("Bài 3: OK")

Bài 1: 2016:12 True True / True False
Bài 2: Day -> [1, 2, 3] | Hour -> [2, 3]
Bài 3: OK


In [36]:
# Bài 4
class LogSystemKeyMineSol:
    SIZES = (10000, 13, 32, 24, 60, 60)

    def __init__(self):
        self.logs = []
        self.div = {g: 1 for g in FIELDS}
        for j, g in enumerate(FIELDS):
            for s in self.SIZES[j + 1:]:
                self.div[g] *= s

    def _key(self, ts):
        k = 0
        for p, size in zip(ts.split(":"), self.SIZES):
            k = k * size + int(p)
        return k

    def put(self, id, timestamp):
        self.logs.append((id, self._key(timestamp)))

    def retrieve(self, start, end, granularity):
        d = self.div[granularity]
        lo, hi = self._key(start) // d, self._key(end) // d
        return [id for id, k in self.logs if lo <= k // d <= hi]


assert norm(run_ops(LogSystemKeyMineSol, ops, args)) == EXPECTED
rng = random.Random(4)
for _ in range(100):
    ops_, args_ = gen_session(rng)
    assert norm(run_ops(LogSystemKeyMineSol, ops_, args_)) == oracle_run(ops_, args_)
print("Bài 4: OK")


# Bài 5
class LogCounterSol:
    def __init__(self):
        self.logs = []

    def put(self, id, timestamp):
        bisect.insort(self.logs, (timestamp, id))

    def count(self, start, end, granularity):
        p = GRAN[granularity]
        lo, hi = start[:p] + MIN_TS[p:], end[:p] + MAX_TS[p:]
        l = bisect.bisect_left(self.logs, (lo, -1))
        r = bisect.bisect_right(self.logs, (hi, INF))
        return max(0, r - l)


rng = random.Random(5)
for _ in range(100):
    logs = [(i + 1, gen_ts(rng, [2016, 2017], [1, 2, 12])) for i in range(30)]
    c = LogCounterSol()
    for id, ts in logs:
        c.put(id, ts)
    s, e = gen_ts(rng, [2016, 2017], [1, 2, 12]), gen_ts(rng, [2016, 2017], [1, 2, 12])   # có thể start > end
    for g in FIELDS:
        assert c.count(s, e, g) == len(oracle(logs, s, e, g))
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (Cách 2, so sánh tiền tố chuỗi):**

```python
class LogSystem:
    def __init__(self):
        self.logs = []
        self.d = {"Year": 4, "Month": 7, "Day": 10, "Hour": 13, "Minute": 16, "Second": 19}

    def put(self, id, timestamp):
        self.logs.append((id, timestamp))

    def retrieve(self, start, end, granularity):
        p = self.d[granularity]
        return [id for id, ts in self.logs if start[:p] <= ts[:p] <= end[:p]]
```

**Công thức nhớ:** *tiền tố dài `4, 7, 10, 13, 16, 19` → `start[:p] <= ts[:p] <= end[:p]`.* Với `bisect`: đệm `lo` bằng `:01:01:00:00:00`, đệm `hi` bằng `:12:31:23:59:59`.

**Checklist trước khi submit:**

- [ ] Dùng đúng độ dài tiền tố cho từng granularity (`4, 7, 10, 13, 16, 19`), áp dụng cho **cả `start`, `end` và log**.
- [ ] Hai đầu **bao gồm** (`<=` ở cả hai phía).
- [ ] Không so từng trường độc lập; so **từ điển** (chuỗi hoặc list/tuple).
- [ ] Nếu dùng `bisect`: đệm **biên dưới** bằng trường nhỏ nhất và **biên trên** bằng trường lớn nhất.
- [ ] Không dùng `datetime` (ngày 31 ở tháng bất kỳ sẽ lỗi).
- [ ] Đã test: log bằng `start`/`end`, `start > end`, cùng timestamp nhiều `id`, chưa có log, mỗi granularity.
- [ ] Thứ tự `id` trả về không bị đề ép: so sánh bằng tập đã sắp xếp khi tự test.
- [ ] Bài Premium: đã đối chiếu đề gốc nếu có tài khoản.